# 05 - Analítica avanzada: clustering

**Proyecto:** Analítica de ventas de videojuegos en Databricks  
**Autor:** Darren J. Blackman M.  
**Asignatura:** Análisis de Datos y Toma de Decisiones en Computación

## Objetivo
Aplicar **K-Means** para segmentar videojuegos según su patrón de ventas regionales. El modelo usa transformaciones logarítmicas para disminuir el efecto de los superventas y estandariza las variables antes de agrupar.

La métrica de evaluación es el **silhouette score**. Se comparan valores de `k` entre 2 y 6 y se elige automáticamente el mejor resultado.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]
ESQUEMA = "default"

def tabla(nombre):
    return f"`{CATALOGO}`.`{ESQUEMA}`.`{nombre}`"

print(f"Catálogo activo: {CATALOGO}")
print(f"Esquema utilizado: {ESQUEMA}")

Catálogo activo: workspace
Esquema utilizado: default


In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

TABLA_LIMPIA = tabla("vgsales_limpio")
TABLA_CLUSTERS = tabla("vgsales_clusters")

df = spark.table(TABLA_LIMPIA)

In [0]:
# Transformación log1p de ventas regionales
regiones = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
df_modelo = df
for c in regiones:
    df_modelo = df_modelo.withColumn(f"log_{c}", F.log1p(F.col(c)))

assembler = VectorAssembler(
    inputCols=[f"log_{c}" for c in regiones],
    outputCol="features_raw"
)
df_vector = assembler.transform(df_modelo)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(df_vector)
df_escalado = scaler_model.transform(df_vector)

In [0]:
# Comparación de k entre 2 y 6
evaluator = ClusteringEvaluator(
    featuresCol="features_scaled",
    predictionCol="Cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

resultados_k = []
modelos = {}
for k in range(2, 7):
    modelo = KMeans(
        k=k,
        seed=42,
        maxIter=50,
        featuresCol="features_scaled",
        predictionCol="Cluster"
    ).fit(df_escalado)
    pred = modelo.transform(df_escalado)
    score = float(evaluator.evaluate(pred))
    resultados_k.append((k, score))
    modelos[k] = modelo

resultados_df = spark.createDataFrame(resultados_k, ["k", "Silhouette"])
display(resultados_df.orderBy(F.desc("Silhouette")))

k,Silhouette
2,0.8864471902897585
3,0.8091334558166342
4,0.7850994058430283
5,0.7362342199011743
6,0.7109552649083678


In [0]:
# Selección y entrenamiento final
mejor_k, mejor_score = max(resultados_k, key=lambda x: x[1])
modelo_final = modelos[mejor_k]
predicciones = modelo_final.transform(df_escalado)

print(f"K seleccionado: {mejor_k}")
print(f"Silhouette score: {mejor_score:.4f}")

K seleccionado: 2
Silhouette score: 0.8864


In [0]:
# Perfil de cada clúster
perfil = (
    predicciones.groupBy("Cluster")
    .agg(
        F.count("*").alias("Cantidad_Juegos"),
        F.round(F.avg("Global_Sales"), 3).alias("Promedio_Global"),
        F.round(F.sum("Global_Sales"), 2).alias("Total_Global"),
        F.round(F.avg("NA_Sales"), 3).alias("Promedio_NA"),
        F.round(F.avg("EU_Sales"), 3).alias("Promedio_EU"),
        F.round(F.avg("JP_Sales"), 3).alias("Promedio_JP"),
        F.round(F.avg("Other_Sales"), 3).alias("Promedio_Otras")
    )
    .orderBy("Promedio_Global")
)
display(perfil)

Cluster,Cantidad_Juegos,Promedio_Global,Total_Global,Promedio_NA,Promedio_EU,Promedio_JP,Promedio_Otras
0,15343,0.303,4653.8,0.153,0.072,0.055,0.023
1,984,4.234,4166.56,2.015,1.332,0.449,0.438


In [0]:
# Etiquetas interpretativas ordenadas por promedio de ventas globales
orden = [r["Cluster"] for r in perfil.select("Cluster").collect()]
if len(orden) == 2:
    nombres = ["Ventas bajas", "Ventas altas"]
elif len(orden) == 3:
    nombres = ["Ventas bajas", "Ventas medias", "Ventas altas"]
else:
    nombres = ["Ventas muy bajas", "Ventas medias", "Ventas altas", "Superventas"]
    nombres += [f"Segmento {i+1}" for i in range(max(0, len(orden)-len(nombres)))]

mapeo = dict(zip(orden, nombres[:len(orden)]))

col_etiqueta = F.lit("Sin etiqueta")
for cluster_id, etiqueta in reversed(list(mapeo.items())):
    col_etiqueta = F.when(
        F.col("Cluster") == int(cluster_id),
        F.lit(etiqueta)
    ).otherwise(col_etiqueta)

predicciones_finales = predicciones.withColumn("Etiqueta_Cluster", col_etiqueta)

display(predicciones_finales.select(
    "Name", "Platform", "Genre", "Year", "Global_Sales", "Cluster", "Etiqueta_Cluster"
).orderBy(F.desc("Global_Sales")).limit(20))

Name,Platform,Genre,Year,Global_Sales,Cluster,Etiqueta_Cluster
Wii Sports,Wii,Sports,2006,82.74,1,Ventas altas
Super Mario Bros.,NES,Platform,1985,40.24,1,Ventas altas
Mario Kart Wii,Wii,Racing,2008,35.82,1,Ventas altas
Wii Sports Resort,Wii,Sports,2009,33.0,1,Ventas altas
Pokemon Red/Pokemon Blue,GB,Role-Playing,1996,31.37,1,Ventas altas
Tetris,GB,Puzzle,1989,30.26,1,Ventas altas
New Super Mario Bros.,DS,Platform,2006,30.01,1,Ventas altas
Wii Play,Wii,Misc,2006,29.02,1,Ventas altas
New Super Mario Bros. Wii,Wii,Platform,2009,28.62,1,Ventas altas
Duck Hunt,NES,Shooter,1984,28.31,1,Ventas altas


In [0]:
# Guardar resultados sin columnas vectoriales
salida = predicciones_finales.drop("features_raw", "features_scaled")
(
    salida.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_CLUSTERS)
)

print(f"Tabla de clústeres creada: {TABLA_CLUSTERS}")

Tabla de clústeres creada: `workspace`.`default`.`vgsales_clusters`



### Interpretación obligatoria

- **K seleccionado:** 2
- **Silhouette score:** 0.8864
- **Clúster de mayor tamaño:** Clúster 0, con 15,343 videojuegos.
- **Clúster con mayor promedio global:** Clúster 1, con un promedio de 4.234 millones de ventas globales por videojuego.

El modelo dividió los videojuegos en dos segmentos claramente diferenciados. El clúster 0 reúne la mayor parte de los registros y presenta un promedio global de 0.303 millones de ventas. Este segmento representa títulos con ventas bajas o moderadas y menor participación promedio en todas las regiones.

El clúster 1 contiene 984 videojuegos y presenta un promedio global de 4.234 millones. Sus ventas promedio son mayores principalmente en Norteamérica y Europa, por lo que corresponde al segmento de videojuegos con ventas altas y mayor alcance internacional.

El silhouette score de 0.8864 indica que los dos grupos presentan una separación clara y que los videojuegos incluidos dentro de cada clúster tienen características de ventas similares.